# Time-fractional diffusion, step by step

We solve the time-fractional heat equation on the unit square,

$$
D_C^{\alpha} u - \kappa\,\Delta u = f
\quad\text{in }\Omega=(0,1)^2,
\qquad u = 0 \text{ on }\partial\Omega,
\qquad u(\cdot,0)=0,
$$

with a Caputo derivative of order $0<\alpha<1$. Replacing $\partial_t$ by
$D_C^\alpha$ changes the character of the problem: the solution at time $t$
depends on its **entire history**.

We build this up in stages, plotting as we go:

1. a graded mesh and its purpose.
2. a 0D warm-up against an exact Mittag-Leffler solution.
3. the 2D problem and its residual.
4. verification against a manufactured solution.
5. separate measurements of memory-mode and timestep error.
6. variable (graded) time steps.

Run this in an activated Firedrake environment with Yonderdrake installed.

> **These notebooks run in a single process.** A Jupyter kernel is one process,
> so everything here executes serially however the environment was launched.
>
> For parallel runs, use `mpiexec -n N python your_script.py`, as in the scripts
> under `demos/`.
> Driving MPI from Jupyter via `ipyparallel` is currently untested.

In [ ]:
import firedrake as fd
import matplotlib.pyplot as plt
import numpy as np
from firedrake import (
    Constant,
    DirichletBC,
    Function,
    SpatialCoordinate,
    TestFunction,
    dx,
    errornorm,
    grad,
    inner,
    norm,
    pi,
    sin,
)
from firedrake.pyplot import tripcolor, triplot

from yonderdrake import BirkSong, CaputoDerivative, FractionalTimeStepper

plt.rcParams["figure.dpi"] = 110
print("yonderdrake", __import__("yonderdrake").__version__)

## 1. A graded mesh

Yonderdrake takes any Firedrake mesh. This coordinate map builds a graded one
without an external mesh generator, and the same construction grades the time
levels in step 6. For $r>1$,

$$
g(\xi)=\begin{cases}\tfrac12(2\xi)^r, & \xi\le\tfrac12,\\[2pt]
1-\tfrac12\bigl(2(1-\xi)\bigr)^r, & \xi>\tfrac12,\end{cases}
$$

applied to each coordinate, clusters cells near both ends of $[0,1]$. Setting
$r=1$ gives back the uniform mesh. Grading is not needed for the smooth
solution used here, but we keep it on to show the construction.

In [ ]:
def graded_unit_square(n, r=2.0):
    "Unit square with cells graded towards the boundary (r=1 is uniform)."
    base = fd.UnitSquareMesh(n, n, diagonal="crossed")
    coordinates = base.coordinates.function_space()
    x, y = SpatialCoordinate(base)

    def grade(xi):
        return fd.conditional(
            fd.le(xi, 0.5),
            0.5 * (2 * xi) ** r,
            1.0 - 0.5 * (2 * (1 - xi)) ** r,
        )

    mapped = Function(coordinates).interpolate(fd.as_vector([grade(x), grade(y)]))
    return fd.Mesh(mapped)


mesh = graded_unit_square(16, r=2.0)

diameters = Function(fd.FunctionSpace(mesh, "DG", 0)).interpolate(fd.CellDiameter(mesh))
smallest, largest = diameters.dat.data_ro.min(), diameters.dat.data_ro.max()
print(f"cells: {mesh.num_cells()}")
print(
    f"cell diameter: min {smallest:.4f}, max {largest:.4f}, "
    f"ratio {largest / smallest:.1f}"
)

figure, axes = plt.subplots(1, 2, figsize=(9, 4))
triplot(graded_unit_square(16, r=1.0), axes=axes[0])
axes[0].set_title("uniform (r = 1)")
triplot(mesh, axes=axes[1])
axes[1].set_title("graded towards the boundary (r = 2)")
for axis in axes:
    axis.set_aspect("equal")
    axis.set_xticks([0, 0.5, 1])
    axis.set_yticks([0, 0.5, 1])
plt.tight_layout()

## 2. Warm-up: one degree of freedom, exact answer

Before touching a PDE, check the machinery on something with a closed form. The
relaxation problem

$$
D_C^{\alpha}u + u = 0, \qquad u(0)=1
$$

has the exact solution $u(t)=E_\alpha(-t^\alpha)$, where $E_\alpha$ is the
Mittag-Leffler function, the fractional analogue of $\exp$. For $\alpha=1$ it
equals $\exp(-t)$. For $\alpha<1$ it decays algebraically.

We solve it on a single-cell mesh, so the only errors are temporal.

In [ ]:
import mpmath as mp

mp.mp.dps = 30


def mittag_leffler(alpha, z, terms=200):
    "E_alpha(z) by its defining series (adequate for the small |z| used here)."
    return float(sum(mp.mpf(z) ** k / mp.gamma(alpha * k + 1) for k in range(terms)))


alpha = 0.6
interval = fd.UnitIntervalMesh(1)
W = fd.FunctionSpace(interval, "CG", 1)
w = Function(W).assign(1.0)
q = TestFunction(W)
time, step = Constant(0.0), Constant(0.02)

relaxation = (inner(CaputoDerivative(w, alpha), q) + inner(w, q)) * dx
relaxation_stepper = FractionalTimeStepper(relaxation, BirkSong(48), time, step, w)

times, computed = [0.0], [1.0]
for _ in range(100):
    relaxation_stepper.advance()
    time.assign(time + step)
    times.append(float(time))
    computed.append(float(w.dat.data_ro[0]))

times = np.array(times)
exact = np.array([mittag_leffler(alpha, -(t**alpha)) if t > 0 else 1.0 for t in times])

figure, axes = plt.subplots(1, 2, figsize=(9, 3.4))
axes[0].plot(times, exact, "k-", label=r"exact $E_\alpha(-t^\alpha)$")
axes[0].plot(times[::5], computed[::5], "o", ms=4, label="BirkSong(48)")
axes[0].plot(times, np.exp(-times), "--", color="0.6", label=r"$e^{-t}$ ($\alpha=1$)")
axes[0].set_xlabel("t")
axes[0].set_ylabel("u")
axes[0].legend()
axes[0].set_title(f"relaxation, alpha = {alpha}")
axes[1].semilogy(times[1:], np.abs(computed[1:] - exact[1:]))
axes[1].set_xlabel("t")
axes[1].set_ylabel("absolute error")
axes[1].set_title("error against the exact solution")
plt.tight_layout()

print(f"max absolute error: {np.max(np.abs(computed - exact)):.3e}")

The fractional solution remains above $e^{-t}$ at $t=2$. The stepper represents
this long memory over the complete time interval.

The error is largest at the first step and then decays. Even with smooth data,
$E_\alpha(-t^\alpha)$ has a weak $t^{\alpha}$ singularity at the origin. A
uniform grid resolves this initial region least accurately. Step 6 applies a
graded time grid there.

## 3. The 2D problem

Two rules govern how you write the residual:

- **the marker wraps the stepped field directly.** `CaputoDerivative(u, alpha)`
  goes on `u`. Ordinary spatial operators go around it. Wrapping a transformed
  expression is not supported.
- **the residual is evaluated at the advanced time.** So a source term written
  in terms of the `Constant` `t` is evaluated at $t+\Delta t$ during the solve.
  You still advance your own `t` after each successful step.

Build `f` from `t`. The stepper evaluates it at `t + dt` during the solve.

We use a manufactured solution so that we know the answer:

$$
u(x,y,t)=t^{2}\sin(\pi x)\sin(\pi y),
\qquad
f=\Bigl[\tfrac{\Gamma(3)}{\Gamma(3-\alpha)}t^{2-\alpha}+2\pi^{2}\kappa\,t^{2}\Bigr]\sin(\pi x)\sin(\pi y),
$$

using $D_C^\alpha t^{p}=\frac{\Gamma(p+1)}{\Gamma(p+1-\alpha)}t^{p-\alpha}$ and
$-\Delta\sin(\pi x)\sin(\pi y)=2\pi^{2}\sin(\pi x)\sin(\pi y)$.

In [ ]:
from math import gamma

alpha, kappa = 0.6, 1.0
final_time, num_steps = 0.5, 50

V = fd.FunctionSpace(mesh, "CG", 2)
x, y = SpatialCoordinate(mesh)
profile = sin(pi * x) * sin(pi * y)

u = Function(V, name="u")
v = TestFunction(V)
t = Constant(0.0)
dt = Constant(final_time / num_steps)
bc = DirichletBC(V, 0.0, "on_boundary")

source = (
    gamma(3.0) / gamma(3.0 - alpha) * t ** (2.0 - alpha) + 2.0 * pi**2 * kappa * t**2
) * profile

residual = (
    inner(CaputoDerivative(u, alpha), v)
    + kappa * inner(grad(u), grad(v))
    - inner(source, v)
) * dx

stepper = FractionalTimeStepper(
    residual,
    BirkSong(64),
    t,
    dt,
    u,
    bcs=bc,
    solver_parameters={"ksp_type": "preonly", "pc_type": "lu"},
)
print(residual.arguments())

Now step, keeping a few snapshots to look at.

In [ ]:
snapshot_times = [0.1, 0.2, 0.35, 0.5]
snapshots = []

for _step_index in range(num_steps):
    stepper.advance()
    t.assign(t + dt)
    if any(abs(float(t) - target) < 1e-9 for target in snapshot_times):
        snapshots.append((float(t), u.copy(deepcopy=True)))

figure, axes = plt.subplots(1, len(snapshots), figsize=(3.2 * len(snapshots), 3.1))
for axis, (snapshot_time, field) in zip(axes, snapshots, strict=False):
    contours = tripcolor(field, axes=axis, cmap="viridis")
    axis.set_title(f"t = {snapshot_time:.2f}")
    axis.set_aspect("equal")
    axis.set_xticks([])
    axis.set_yticks([])
    figure.colorbar(contours, ax=axis, fraction=0.046)
plt.tight_layout()

## 4. Is it right?

Compare the computed field with the manufactured solution at the final time.
Integrating against the exact expression at high quadrature degree keeps the
spatial discretization error in the number, so this is the full error of the
computed field. Section 5 separates the
contributions.

In [ ]:
exact = float(t) ** 2 * profile
accurate = dx(degree=10)
error = fd.assemble((exact - u) ** 2 * accurate) ** 0.5
scale = fd.assemble(exact**2 * accurate) ** 0.5
print(f"final time       : {float(t):.3f}")
print(f"relative L2 error: {error / scale:.3e}")

## 5. Separating the two error sources

A fractional solve has more independent knobs than a classical one, and they must
be refined **separately**:

- the **mode count** controls how well the diffusive representation approximates
  the memory kernel. It is a quadrature parameter, independent of the time grid.
- the **timestep** controls the temporal discretization.

Increase the mode count to resolve the kernel. Reduce `dt` to resolve the
timestep. Each sweep below refines one control while holding the other fixed,
and measures against a reference that differs *only* in the control being
refined. Measuring against the exact solution instead would fold in the spatial
discretization error, which on this mesh is larger than either temporal error
and would flatten both curves onto it.

In [ ]:
def solve_to_final_time(num_modes, steps, space):
    "Run the manufactured problem and return the field at the final time."
    field = Function(space)
    test = TestFunction(space)
    clock = Constant(0.0)
    increment = Constant(final_time / steps)
    boundary = DirichletBC(space, 0.0, "on_boundary")
    forcing = (
        gamma(3.0) / gamma(3.0 - alpha) * clock ** (2.0 - alpha)
        + 2.0 * pi**2 * kappa * clock**2
    ) * profile
    form = (
        inner(CaputoDerivative(field, alpha), test)
        + kappa * inner(grad(field), grad(test))
        - inner(forcing, test)
    ) * dx
    run = FractionalTimeStepper(
        form,
        BirkSong(num_modes),
        clock,
        increment,
        field,
        bcs=boundary,
        solver_parameters={"ksp_type": "preonly", "pc_type": "lu"},
    )
    for _ in range(steps):
        run.advance()
        clock.assign(clock + increment)
    return field


def difference(field, reference):
    "Relative L2 difference between two fields on the same space."
    return errornorm(reference, field, "L2") / norm(reference)


# Refining modes against a many-mode reference at the same dt leaves only the
# representation error; refining dt against a small-dt reference at the same
# mode count leaves only the time-discretization error.
mode_counts = [2, 4, 8, 16, 32]
mode_reference = solve_to_final_time(128, 200, V)
mode_errors = [
    difference(solve_to_final_time(m, 200, V), mode_reference) for m in mode_counts
]

step_counts = [10, 20, 40, 80, 160]
step_reference = solve_to_final_time(64, 640, V)
step_errors = [
    difference(solve_to_final_time(64, s, V), step_reference) for s in step_counts
]

figure, axes = plt.subplots(1, 2, figsize=(9, 3.4))
axes[0].loglog(mode_counts, mode_errors, "o-")
axes[0].set_xlabel("memory modes")
axes[0].set_ylabel(r"relative $L^2$ difference")
axes[0].set_title("refine modes (dt fixed small)")
axes[1].loglog(final_time / np.array(step_counts), step_errors, "o-")
axes[1].set_xlabel(r"$\Delta t$")
axes[1].set_ylabel(r"relative $L^2$ difference")
axes[1].set_title("refine dt (64 modes)")
for axis in axes:
    axis.grid(True, which="both", alpha=0.3)
plt.tight_layout()

for count, value in zip(mode_counts, mode_errors, strict=True):
    print(f"{count:3d} modes -> {value:.3e}")
for count, value in zip(step_counts, step_errors, strict=True):
    print(f"dt = {final_time / count:.5f} -> {value:.3e}")

## 6. Variable time steps, and the fix for step 2

The stepper accepts a changing `dt`, and step 2 showed us why that matters: the
error was concentrated at $t\approx0$, where $E_\alpha(-t^\alpha)$ has its
$t^{\alpha}$ singularity. Clustering time levels near the origin is the temporal
analogue of the graded mesh from step 1. Using

$$t_n = T\,(n/N)^{r},\qquad r>1,$$

we keep the same number of steps and simply place them better. Let us return to
the relaxation problem and sweep the grading exponent.

In [ ]:
def relaxation_errors(levels, num_modes=48):
    "Solve D^alpha w + w = 0 on given time levels; return |error| at each level."
    space = fd.FunctionSpace(fd.UnitIntervalMesh(1), "CG", 1)
    field = Function(space).assign(1.0)
    test = TestFunction(space)
    clock, increment = Constant(0.0), Constant(1.0)
    form = (inner(CaputoDerivative(field, alpha), test) + inner(field, test)) * dx
    run = FractionalTimeStepper(form, BirkSong(num_modes), clock, increment, field)
    computed = [1.0]
    for previous, current in zip(levels[:-1], levels[1:], strict=False):
        increment.assign(current - previous)
        run.advance()
        clock.assign(current)
        computed.append(float(field.dat.data_ro[0]))
    truth = np.array(
        [mittag_leffler(alpha, -(s**alpha)) if s > 0 else 1.0 for s in levels]
    )
    return np.abs(np.array(computed) - truth)


horizon, count = 2.0, 100
gradings = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5]
worst = []
for grading in gradings:
    levels = horizon * (np.arange(count + 1) / count) ** grading
    worst.append(relaxation_errors(levels).max())

uniform_levels = horizon * np.arange(count + 1) / count
graded_levels = horizon * (np.arange(count + 1) / count) ** 3.0
uniform_error = relaxation_errors(uniform_levels)
graded_error = relaxation_errors(graded_levels)

figure, axes = plt.subplots(1, 2, figsize=(9, 3.4))
axes[0].semilogy(gradings, worst, "o-")
axes[0].set_xlabel("grading exponent r")
axes[0].set_ylabel("max absolute error")
axes[0].set_title(f"same {count} steps, better placed")
axes[0].grid(True, alpha=0.3)
axes[1].semilogy(uniform_levels[1:], uniform_error[1:], label="uniform (r = 1)")
axes[1].semilogy(graded_levels[1:], graded_error[1:], label="graded (r = 3)")
axes[1].set_xlabel("t")
axes[1].set_ylabel("absolute error")
axes[1].legend()
axes[1].set_title("error against time")
plt.tight_layout()

print(f"uniform  max error: {uniform_error.max():.3e}")
print(f"graded   max error: {graded_error.max():.3e}")
print(f"improvement factor: {uniform_error.max() / graded_error.max():.1f}x")

Same solver, same number of steps, same mode count. Only the placement of the
time levels changed. Optimal grading exponents for L1-type schemes are studied in
the literature. Here the graded grid reduces the error near the initial layer.

The same mechanics apply to the PDE. Our manufactured solution is deliberately
smooth ($t^{2}$) and so has no initial layer to repair, which makes it a good
check that a non-uniform grid does not by itself degrade an already-clean result.

In [ ]:
def graded_time_grid(final, steps, grading):
    "Time levels clustered near t = 0 for grading > 1."
    return final * (np.arange(steps + 1) / steps) ** grading


field = Function(V)
test = TestFunction(V)
clock = Constant(0.0)
increment = Constant(1.0)
boundary = DirichletBC(V, 0.0, "on_boundary")
forcing = (
    gamma(3.0) / gamma(3.0 - alpha) * clock ** (2.0 - alpha)
    + 2.0 * pi**2 * kappa * clock**2
) * profile
form = (
    inner(CaputoDerivative(field, alpha), test)
    + kappa * inner(grad(field), grad(test))
    - inner(forcing, test)
) * dx
graded_stepper = FractionalTimeStepper(
    form,
    BirkSong(64),
    clock,
    increment,
    field,
    bcs=boundary,
    solver_parameters={"ksp_type": "preonly", "pc_type": "lu"},
)

levels = graded_time_grid(final_time, 50, grading=2.0)
for previous, current in zip(levels[:-1], levels[1:], strict=True):
    increment.assign(current - previous)
    graded_stepper.advance()
    clock.assign(current)

# Measured exactly as in section 4, so the two numbers are comparable.
truth = float(clock) ** 2 * profile
graded_error = fd.assemble((truth - field) ** 2 * accurate) ** 0.5
graded_scale = fd.assemble(truth**2 * accurate) ** 0.5
print(f"steps taken        : {len(levels) - 1}")
print(f"smallest dt        : {np.diff(levels).min():.5f}")
print(f"largest dt         : {np.diff(levels).max():.5f}")
print(f"relative L2 error  : {graded_error / graded_scale:.3e}")

plt.figure(figsize=(6, 2.2))
plt.plot(levels, np.zeros_like(levels), "|", ms=14)
plt.yticks([])
plt.xlabel("t")
plt.title("graded time levels, clustered near t = 0")
plt.tight_layout()

## Where to go next

- Swap the representation: `Diethelm2008(64)` in place of `BirkSong(64)` changes one
  argument. Equal mode counts do not imply equal accuracy, so compare against a
  shared reference at fixed cost. (Both are the same Gauss-Jacobi construction at
  a different exponent: they are `Cayley(64, power=2)` and `Cayley(64, power=4)`.
  A larger exponent spans more decades of relaxation rate and resolves each one
  less finely, and `Cayley` will size it for you from a declared `t_final` and
  `min_step`.)
- Swap the derivative: `RiemannLiouvilleDerivative` adds an exact initial-trace
  term and does *not* annihilate a nonzero constant.
- Replace the classical Laplacian with a **fractional** one. The companion
  notebook is `02-spectral-fractional-laplacian.ipynb`.
- The library documentation covers method selection, the defaults policy, and how
  each representation relates to its source paper.